In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install lancedb chromadb langchain langchain_openai langchain_core langchain_community openai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 35.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 61.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.8/450.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.

In [2]:
import os
import pandas as pd
import lancedb

from langchain.document_loaders import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import LanceDB, Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA, LLMChain, ConversationalRetrievalChain
from langchain_openai import ChatOpenAI, OpenAI
from langchain.memory import ConversationBufferMemory

In [ ]:
os.environ['OPENAI_API_KEY'] = "api-key"
os.environ['OPENAI_API_BASE'] = "https://openai.vocareum.com/v1"

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = "api-key", base_url = "https://openai.vocareum.com/v1")
prompt = """Generate 10 synthetic real estate data listings. Return ONLY in valid csv format. 
Columns should be id, city, type, bedrooms, bathrooms, price, description.
Rules:
1. id: 1-10
2. city: Mention cities of Odisha
3. Type: Mention 5 property types
4. Bedrooms: 1-5
5. Bathrooms: 1-3
6. Description: mention additionally atmost 3 features like modern kitchen, balcony, garden, quiet neighbourhood, near shopping malls. write them in sentence.
"""
response = client.chat.completions.create(
    model = "gpt-3.5-turbo",
    temperature = 0.5,
    messages = [{"role":"user", "content":prompt}]
)

csv_txt = response.choices[0].message.content.strip()

with open ("real_estate_listings.csv", "w", encoding = "utf-8") as f:
    f.write(csv_txt)

print("Generation Successful!")

Generation Successful!


In [5]:
df = pd.read_csv("real_estate_listings.csv")
df.head(10)

,id,city,type,bedrooms,bathrooms,price,description
0,1,Bhubaneswar,Apartment,2,1,50000,"Modern kitchen, balcony, near shopping malls"
1,2,Cuttack,House,3,2,75000,"Balcony, garden, quiet neighbourhood"
2,3,Puri,Villa,4,3,100000,"Modern kitchen, garden, near shopping malls"
3,4,Rourkela,Condo,1,1,30000,"Balcony, quiet neighbourhood, near shopping malls"
4,5,Sambalpur,Townhouse,2,2,60000,"Modern kitchen, garden, quiet neighbourhood"
5,6,Berhampur,Apartment,3,1,70000,"Balcony, near shopping malls, quiet neighbourhood"
6,7,Balasore,House,4,2,90000,"Modern kitchen, garden, near shopping malls"
7,8,Jharsuguda,Villa,5,3,120000,"Balcony, quiet neighbourhood, near shopping malls"
8,9,Baripada,Condo,2,1,45000,"Modern kitchen, garden, near shopping malls"
9,10,Bargarh,Townhouse,3,2,80000,"Balcony, quiet neighbourhood, near shopping malls"


In [6]:
docs = CSVLoader(file_path = "real_estate_listings.csv").load()

In [ ]:
emb = OpenAIEmbeddings(openai_api_key = "api-key", openai_api_base="https://openai.vocareum.com/v1")
db = lancedb.connect("./lancedb")

vector_store = LanceDB.from_documents(
    docs,
    embedding = emb,
    connection = db,
    table_name = "real_estate"
)
print(f"Successfully stored {len(docs)} complete listings in vector database")

/tmp/ipykernel_48/2257979753.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  emb = OpenAIEmbeddings(openai_api_key = "voc-10562236181266774374597690cf295afbd81.25272991", openai_api_base="https://openai.vocareum.com/v1")


Successfully stored 10 complete listings in vector database


In [12]:
def semantic_search_on_listings(query, k=5):
    
    results = vector_store.similarity_search(query, k=k)
    
    print(f"Semantic Search Results for: '{query}'\n")
    print("-" * 80)
    
    for i, r in enumerate(results, 1):
        print(f"Result {i}:")
        print(r.page_content)
        print("-" * 60)
    
    return results

query1 = "quiet neighborhood and modern kitchen"
results = semantic_search_on_listings(query1)

query2 = "apartment in Bhubaneswar with balcony"
results2 = semantic_search_on_listings(query2, k=3)

Semantic Search Results for: 'quiet neighborhood and modern kitchen'

--------------------------------------------------------------------------------
Result 1:
id: 1
city: Bhubaneswar
type: Apartment
bedrooms: 2
bathrooms: 1
price: 50000
description: Modern kitchen, balcony, near shopping malls
------------------------------------------------------------
Result 2:
id: 5
city: Sambalpur
type: Townhouse
bedrooms: 2
bathrooms: 2
price: 60000
description: Modern kitchen, garden, quiet neighbourhood
------------------------------------------------------------
Result 3:
id: 3
city: Puri
type: Villa
bedrooms: 4
bathrooms: 3
price: 100000
description: Modern kitchen, garden, near shopping malls
------------------------------------------------------------
Result 4:
id: 7
city: Balasore
type: House
bedrooms: 4
bathrooms: 2
price: 90000
description: Modern kitchen, garden, near shopping malls
------------------------------------------------------------
Result 5:
id: 9
city: Baripada
type: Condo


In [23]:
from langchain.schema import Document

def listing_descriptions(listings, preferences):
    listings1 = []
    
    for r in listings:
        original_content = r.page_content

        prompt = f"""
        You are a professional real estate agent. Enhance this property description to better match the buyer's preferences.
        
        Buyer Preferences: {preferences}
        
        Original listing:
        {original_content}
        
        RULES:
        1. Keep ALL factual information exactly the same (id, city, type, bedrooms, bathrooms, price)
        2. ONLY enhance the description field to make it more appealing based on the buyer's preferences
        3. Maintain a professional, marketing tone
        4. Do not add or change any factual details
        5. Make the description more personalized and engaging
        
        Return the listing in the EXACT same format as the original, only changing the description field.
        """
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0.5,
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response.choices[0].message.content.strip()
        
        doc = Document(
            page_content=content,
            metadata=r.metadata
        )
        
        listings1.append({
            'original': r,
            'enhanced': doc
        })
    
    return listings1

def display_before_after_comparison(listings1):
    for i, item in enumerate(listings1, 1):
        print(f"\n{'-'*100}")
        print(f"Property {i} - Before & After Personalization")
        print(f"{'-'*100}")
        
        print("\n Original Listing:")
        print("-" * 50)
        print(item['original'].page_content)
        
        print("\n Listing:")
        print("-" * 50)
        print(item['enhanced'].page_content)
        print("\n" + "-"*100 + "\n")


def personalized_property_search(user_preferences, k=3):
    """Complete RAG pipeline with personalization"""
    print(f"Searching for properties matching: {user_preferences}")
    print("-" * 80)
    
    search_results = vector_store.similarity_search(user_preferences, k=k)
    
    print(f"Found {len(search_results)} matching properties")
    print("=" * 80)
    
    results = listing_descriptions(search_results, user_preferences)
    
    display_before_after_comparison(results)
    
    return results

user_prefs = "A young working couple looking for a quiet neighborhood with modern amenities and nearby shopping"
personalized_results = personalized_property_search(user_prefs)

Searching for properties matching: A young working couple looking for a quiet neighborhood with modern amenities and nearby shopping
--------------------------------------------------------------------------------
Found 3 matching properties

----------------------------------------------------------------------------------------------------
Property 1 - Before & After Personalization
----------------------------------------------------------------------------------------------------

 Original Listing:
--------------------------------------------------
id: 1
city: Bhubaneswar
type: Apartment
bedrooms: 2
bathrooms: 1
price: 50000
description: Modern kitchen, balcony, near shopping malls

 Listing:
--------------------------------------------------
id: 1
city: Bhubaneswar
type: Apartment
bedrooms: 2
bathrooms: 1
price: 50000
description: Welcome to your modern oasis in the heart of Bhubaneswar! This apartment features a sleek kitchen perfect for whipping up meals together, a cozy balcon

In [25]:
def conversational_agent():
    
    llm = ChatOpenAI(
        model="gpt-3.5-turbo", 
        temperature=0.3
    )
    
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )
    
    retriever = vector_store.as_retriever(search_kwargs={"k": 5})
    
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        chain_type="stuff",
        verbose=False
    )
    
    print("Personalized Real Estate Agent Initialized! Type 'quit' to exit.")
    
    
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        
        try:
            search_results = vector_store.similarity_search(user_input, k=2)
            if search_results:
                print(f"\nFound {len(search_results)} properties matching your criteria:")
                print("=" * 80)
                
                listings = listing_descriptions(search_results, user_input)
                
                display_before_after_comparison(listings)
                
                result = qa_chain.invoke({'question': user_input})
                print(f"Agent Summary: {result['answer']}\n")
                
                
        except Exception as e:
            print("Error occurred")
            break

conversational_agent()

Personalized Real Estate Agent Initialized! Type 'quit' to exit.


You:  i am a student who needs a property with quiet neighbourhood



Found 2 properties matching your criteria:

----------------------------------------------------------------------------------------------------
Property 1 - Before & After Personalization
----------------------------------------------------------------------------------------------------

 Original Listing:
--------------------------------------------------
id: 5
city: Sambalpur
type: Townhouse
bedrooms: 2
bathrooms: 2
price: 60000
description: Modern kitchen, garden, quiet neighbourhood

 Listing:
--------------------------------------------------
id: 5
city: Sambalpur
type: Townhouse
bedrooms: 2
bathrooms: 2
price: 60000
description: Perfect townhouse for a student seeking a peaceful retreat in a quiet neighbourhood. This modern property features a renovated kitchen, a lovely garden, and a serene environment ideal for studying and relaxation. Don't miss out on this ideal student haven!

------------------------------------------------------------------------------------------------

You:  quit


Goodbye!
